# KPI de Matrícula (punto 6.2)

Calcula los KPI del grupo Matrícula a partir de `data/integrado/`. Las funciones son puras (reciben y devuelven DataFrames); la lectura de archivos se hace solo en la celda de carga.

KPI de este notebook:
- Estudiantes activos: matriculados y no matriculados (alerta de no inscriptos pero activos)
- Gráfico de situación académica general y por asignatura

## 1. Carga de datos

In [ ]:
from pathlib import Path

import pandas as pd

DATA = Path("../../data")
df_participacion = pd.read_parquet(DATA / "integrado" / "participacion_integrado.parquet")
df_cruce = pd.read_parquet(DATA / "integrado" / "cruce_matriculados_integrado.parquet")

CLAVES = ["materia", "usuario"]

## 2. Estudiantes activos: matriculados y no matriculados

Clasifica a los estudiantes **activos** de cada asignatura según figuren o no en el listado oficial de matriculados. Los no matriculados activos son la alerta: usan el aula sin estar inscriptos.

**Definiciones**
- **Activo:** tuvo al menos una interacción en el aula (`interacciones > 0` en el reporte de participación). Estar listado en el reporte no alcanza: el reporte incluye a todo el padrón de la plataforma en cada aula, incluso a quienes nunca entraron.
- **Matriculado:** figura en el listado oficial de matriculados de esa asignatura (`matriculado` del cruce).

In [ ]:
def estudiantes_activos(participacion: pd.DataFrame, cruce: pd.DataFrame) -> pd.DataFrame:
    """Estudiantes activos de cada asignatura y si están matriculados.

    Devuelve una fila por estudiante activo y asignatura: materia, usuario, matriculado.
    """
    activos = participacion.loc[participacion["interacciones"] > 0, CLAVES]
    res = activos.merge(cruce[CLAVES + ["matriculado"]], on=CLAVES, how="left")
    res["matriculado"] = res["matriculado"].fillna(False).astype(bool)
    return res.sort_values(CLAVES).reset_index(drop=True)


def resumen_activos_por_matricula(activos: pd.DataFrame) -> pd.DataFrame:
    """KPI: cantidad de estudiantes activos matriculados y no matriculados, por asignatura.

    Devuelve una fila por asignatura: materia, matriculados_activos, no_matriculados_activos.
    """
    return (
        activos.groupby(["materia", "matriculado"]).size().unstack(fill_value=0)
        .reindex(columns=[True, False], fill_value=0)
        .set_axis(["matriculados_activos", "no_matriculados_activos"], axis=1)
        .reset_index()
        .rename_axis(columns=None)
    )

In [ ]:
activos = estudiantes_activos(df_participacion, df_cruce)
kpi_activos_matricula = resumen_activos_por_matricula(activos)
kpi_activos_matricula

Listado de los **no matriculados activos** (la alerta), por asignatura:

In [ ]:
activos[~activos["matriculado"]][CLAVES]

### Control

Las cantidades coinciden con el resumen de la integración (`resumen_integracion.parquet`).

In [ ]:
resumen_integracion = pd.read_parquet(DATA / "integrado" / "resumen_integracion.parquet").set_index("materia")
comparacion = kpi_activos_matricula.set_index("materia")[["no_matriculados_activos"]].join(
    resumen_integracion[["activos_no_inscriptos"]]
)
print("Coincide con la integración:", bool((comparacion["no_matriculados_activos"] == comparacion["activos_no_inscriptos"]).all()))
comparacion

## 3. Situación académica general y por asignatura

Muestra cómo se distribuyen los estudiantes **matriculados** según su situación académica en cada asignatura y en el conjunto de todas. Es la mirada de matrícula sobre el rendimiento: además de la condición de quienes ya rindieron, deja ver cuántos todavía no se presentaron a ninguna evaluación y cuántos dieron de baja su inscripción.

**Situaciones** (cada estudiante tiene una por asignatura):

| Situación | Criterio |
|---|---|
| `baja` | La inscripción figura con estado *Baja*. Tiene prioridad sobre las demás. |
| `no se presentó` | Matriculado que no rindió ningún instrumento. |
| `promocionado`, `regular`, `libre` | Condición según su nota (porcentaje de los puntos rendidos) y los umbrales de la asignatura definidos en `kpis_calificaciones.py` (porcentaje del total posible). |

La condición es el **desempeño actual**, no una proyección de la condición final: a medida que se carguen más instrumentos, los estudiantes pasan de *no se presentó* a las demás situaciones sin cambiar el cálculo. El resultado *general* suma todos los pares estudiante-asignatura, por lo que un estudiante que cursa varias asignaturas cuenta una vez en cada una.

In [ ]:
from kpis_calificaciones import condicion_por_estudiante

df_calificaciones = pd.read_parquet(DATA / "integrado" / "calificaciones_integrado.parquet")

SITUACIONES = ["promocionado", "regular", "libre", "no se presentó", "baja"]

In [ ]:
def situacion_academica(calificaciones: pd.DataFrame, cruce: pd.DataFrame) -> pd.DataFrame:
    """Situación académica de cada estudiante matriculado, por asignatura.

    Devuelve una fila por estudiante y asignatura: materia, usuario, situacion.
    """
    matriculados = cruce.loc[cruce["matriculado"], CLAVES + ["estado_insc"]]
    condicion = condicion_por_estudiante(calificaciones, cruce)[CLAVES + ["condicion"]]
    res = matriculados.merge(condicion, on=CLAVES, how="left")
    res["situacion"] = res["condicion"].fillna("no se presentó")
    res.loc[res["estado_insc"] == "Baja", "situacion"] = "baja"
    return res[CLAVES + ["situacion"]].sort_values(CLAVES).reset_index(drop=True)


def resumen_situacion_academica(situacion: pd.DataFrame) -> pd.DataFrame:
    """KPI: cantidad y porcentaje de estudiantes por situación académica, por asignatura y en general.

    Devuelve (una fila por asignatura y situación, incluyendo las situaciones sin estudiantes, y una
    asignatura "general" con el total): materia, situacion, estudiantes, porcentaje. Sirve directo
    para el gráfico de dona o de barras de Streamlit.
    """
    conteo = (
        situacion.groupby(["materia", "situacion"]).size().unstack(fill_value=0)
        .reindex(columns=SITUACIONES, fill_value=0)
    )
    conteo.loc["general"] = conteo.sum()
    res = conteo.stack().rename("estudiantes").reset_index()
    res["porcentaje"] = (res["estudiantes"] / res.groupby("materia")["estudiantes"].transform("sum") * 100).round(2)
    return res

In [ ]:
situacion = situacion_academica(df_calificaciones, df_cruce)
kpi_situacion = resumen_situacion_academica(situacion)
kpi_situacion.pivot(index="materia", columns="situacion", values="estudiantes")[SITUACIONES]

In [ ]:
kpi_situacion.pivot(index="materia", columns="situacion", values="porcentaje")[SITUACIONES]

### Control

La suma por asignatura coincide con la cantidad de matriculados, y la cantidad de bajas con el listado de matrícula.

In [ ]:
matriculados_por_materia = df_cruce[df_cruce["matriculado"]].groupby("materia").size()
totales = kpi_situacion[kpi_situacion["materia"] != "general"].groupby("materia")["estudiantes"].sum()
print("La suma coincide con los matriculados:", bool((totales == matriculados_por_materia).all()))
print("Bajas:", int((situacion["situacion"] == "baja").sum()), "| con estado Baja en el listado:", int((df_cruce["estado_insc"] == "Baja").sum()))

### Gráfico

Barras horizontales al 100 % para cada asignatura y para el total general. La imagen se guarda en `data/kpis/situacion_academica.png`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

COLORES_SITUACION = {"promocionado": "#6B8F71", "regular": "#E0A458", "libre": "#B5523B",
                     "no se presentó": "#8A9BA8", "baja": "#4A5A64"}


def graficar_situacion_academica(resumen: pd.DataFrame, ruta: Path | None = None):
    """Barras horizontales apiladas (100 %) de la situación académica por asignatura y en general."""
    porcentajes = resumen.pivot(index="materia", columns="situacion", values="porcentaje")[SITUACIONES]
    orden = [m for m in porcentajes.index if m != "general"] + ["general"]
    porcentajes = porcentajes.loc[orden]
    etiquetas = [m.capitalize() for m in porcentajes.index]

    fig, eje = plt.subplots(figsize=(9, 3.6))
    izquierda = np.zeros(len(porcentajes))
    for situacion_ in SITUACIONES:
        valores = porcentajes[situacion_].to_numpy()
        barras = eje.barh(etiquetas, valores, left=izquierda, color=COLORES_SITUACION[situacion_], label=situacion_)
        eje.bar_label(barras, labels=[f"{v:.0f} %" if v >= 4 else "" for v in valores], label_type="center",
                      fontsize=9, color="white")
        izquierda += valores
    eje.invert_yaxis()
    eje.set_xlim(0, 100)
    eje.set_xlabel("Porcentaje de matriculados")
    eje.set_title("Situación académica de los matriculados", loc="left")
    eje.spines[["top", "right"]].set_visible(False)
    eje.legend(loc="upper center", bbox_to_anchor=(0.5, -0.22), ncol=len(SITUACIONES), frameon=False)
    fig.tight_layout()
    if ruta is not None:
        fig.savefig(ruta, dpi=150, bbox_inches="tight")
    return fig


fig = graficar_situacion_academica(kpi_situacion, DATA / "kpis" / "situacion_academica.png")
plt.show()